In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lyk1zm/llm-classification-finetuning/sample_submission.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning/train.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning/test.csv
/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv


In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss

SEED = 42

# Исправленный путь
DATA_DIR = Path("/kaggle/input/competitions/llm-classification-finetuning")

# Теперь всё заработается
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())

Train: (57477, 9)
Test: (3, 4)
Train columns: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']
Test columns: ['id', 'prompt', 'response_a', 'response_b']


In [3]:
TARGET_COLUMNS = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]

targets = train[TARGET_COLUMNS]

assert targets.isin([0, 1]).all().all(), "Неожиданные значения меток"
assert targets.sum(axis=1).eq(1).all(), "Должен быть ровно один класс"
assert train["id"].is_unique, "Найдены повторяющиеся id"

train["label"] = targets.to_numpy().argmax(axis=1)

display(
    train["label"]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "A", 1: "B", 2: "Tie"})
    .to_frame("Доля")
)

,Доля
label,
A,0.349079
B,0.341911
Tie,0.309011


In [4]:
TEXT_COLUMNS = ["prompt", "response_a", "response_b"]


def parse_turns(value):
    turns = json.loads(value)

    if not isinstance(turns, list):
        raise ValueError("Ожидался список реплик")

    if any(turn is not None and not isinstance(turn, str) for turn in turns):
        raise ValueError("Встретилась реплика неожиданного типа")

    return ["" if turn is None else turn for turn in turns]


for column in TEXT_COLUMNS:
    train[f"{column}_turns"] = train[column].map(parse_turns)

    # Плоский текст пригодится для EDA и TF-IDF.
    train[f"{column}_text"] = train[f"{column}_turns"].map(
        lambda turns: "\n\n".join(turns)
    )

    train[f"{column}_chars"] = train[f"{column}_text"].str.len()

display(
    train[[f"{column}_chars" for column in TEXT_COLUMNS]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(1)
)

,prompt_chars,response_a_chars,response_b_chars
count,57477.0,57477.0,57477.0
mean,352.6,1330.6,1337.4
std,1025.5,1462.5,1484.3
min,3.0,0.0,0.0
50%,91.0,1036.0,1044.0
90%,758.4,2703.0,2696.0
95%,1419.0,3586.0,3576.2
99%,4699.4,6787.0,6738.5
max,32837.0,53333.0,52433.0


In [5]:
def make_prompt_group(turns):
    normalized = [" ".join(turn.split()) for turn in turns]
    return json.dumps(normalized, ensure_ascii=False)


groups = train["prompt_turns"].map(make_prompt_group)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

train_idx, valid_idx = next(
    splitter.split(
        X=train,
        y=train["label"],
        groups=groups,
    )
)

train_part = train.iloc[train_idx].copy()
valid_part = train.iloc[valid_idx].copy()

assert set(groups.iloc[train_idx]).isdisjoint(
    set(groups.iloc[valid_idx])
)

print("Обучение:", len(train_part))
print("Валидация:", len(valid_part))
print("Уникальных групп запросов:", groups.nunique())

display(
    pd.DataFrame({
        "train": train_part["label"].value_counts(normalize=True),
        "valid": valid_part["label"].value_counts(normalize=True),
    }).sort_index()
)

# Сохраняем разбиение для следующих экспериментов.
split_info = train[["id"]].copy()
split_info["split"] = "train"
split_info.loc[valid_part.index, "split"] = "valid"
split_info.to_csv("split.csv", index=False)

Обучение: 45599
Валидация: 11878
Уникальных групп запросов: 51444


,train,valid
label,,
0,0.349328,0.348123
1,0.342617,0.339199
2,0.308055,0.312679


In [6]:
class_priors = (
    train_part["label"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    .to_numpy()
)

valid_probabilities = np.tile(
    class_priors,
    (len(valid_part), 1),
)

score = log_loss(
    valid_part["label"],
    valid_probabilities,
    labels=[0, 1, 2],
)

print("Вероятности [A, B, Tie]:", class_priors)
print(f"Prior baseline log loss: {score:.5f}")

Вероятности [A, B, Tie]: [0.34932784 0.34261716 0.308055  ]
Prior baseline log loss: 1.09764
